# Lesson 01 - Introduction to AI Agents

Welcome to the first lesson in the **AI Agents for Beginners** course!

An **AI agent** is a program that uses a large language model (LLM) as its reasoning engine and can take *actions* in the real world — calling APIs, querying databases, or running code — to accomplish a goal on behalf of a user.

In this notebook you will build your first agent: a **Travel Agent** that recommends vacation destinations. Along the way you will learn how to:

1. Connect to Azure AI Foundry Agent Service using the **Microsoft Agent Framework**.
2. Give the agent a **tool** — a plain Python function it can call.
3. Run the agent and inspect its response.
4. Stream the agent's response token-by-token.

## Setup

Before running this notebook, make sure you have:

1. **An Azure AI Foundry project** with a deployed chat model (e.g. `gpt-4o-mini`).
2. **Logged in with the Azure CLI** — run `az login` in your terminal.
3. **The Microsoft Agent Framework installed** — the cell below will install it.

The notebook uses `AzureCliCredential` to authenticate, so your `az login` session must be active.

In [1]:
import subprocess
import sys

print("Installing required packages...")
subprocess.check_call([
    sys.executable,
    "-m",
    "pip",
    "install",
    "-q",
    "agent-framework",
    "agent-framework-foundry",
    "azure-identity",
])
print("✓ Packages installed successfully!")
print("\n⚠️  Now restart the Jupyter kernel:")
print("  - In VS Code: Click the 'Restart Kernel' button (or Ctrl+Shift+P > Restart Kernel)")
print("  - Then run the next cells")

Installing required packages...
✓ Packages installed successfully!

⚠️  Now restart the Jupyter kernel:
  - In VS Code: Click the 'Restart Kernel' button (or Ctrl+Shift+P > Restart Kernel)
  - Then run the next cells


## Creating Your First Agent

To build an agent with the current **Microsoft Agent Framework** flow in this environment, you need:

1. **A Foundry Chat Client** — `FoundryChatClient` with your project endpoint and model deployment.
2. **Tool functions** — decorated with `@tool` to expose them to the agent.
3. **An Agent wrapper** — created via `chat_client.as_agent(...)`.

Then run with `await agent.run(...)` and read `result.text`.

If your deployment does not support tool-calling parameters, the code cell will automatically fallback to a no-tool mode so you can still complete the lesson.

In [1]:
from typing import List

# Import the tool decorator
try:
    from agent_framework import tool
except ImportError:
    print("✗ Could not import tool decorator")
    raise

@tool(approval_mode="never_require")
def get_destinations() -> List[str]:
    """Return a curated list of popular vacation destinations."""
    return [
        "Barcelona",
        "Paris",
        "Berlin",
        "Tokyo",
        "Sydney",
        "New York City",
        "Cairo",
        "Cape Town",
        "Rio de Janeiro",
        "Bali",
    ]

print("✓ get_destinations tool defined with @tool decorator")

✓ get_destinations tool defined with @tool decorator


In [2]:
import os
from azure.identity.aio import AzureCliCredential

print("Creating FoundryChatClient and Agent...")
try:
    from agent_framework_foundry import FoundryChatClient

    project_endpoint = os.getenv("AZURE_AI_PROJECT_ENDPOINT") or os.getenv("FOUNDRY_PROJECT_ENDPOINT")
    if not project_endpoint:
        raise ValueError(
            "Missing project endpoint. Set AZURE_AI_PROJECT_ENDPOINT (or FOUNDRY_PROJECT_ENDPOINT) in your environment."
        )

    model_name = os.getenv("AZURE_AI_MODEL_DEPLOYMENT_NAME")
    if not model_name:
        raise ValueError(
            "Missing model deployment name. Set AZURE_AI_MODEL_DEPLOYMENT_NAME in your environment."
        )
    print(f"Using model deployment: {model_name}")

    credential = AzureCliCredential()
    chat_client = FoundryChatClient(
        project_endpoint=project_endpoint,
        model=model_name,
        credential=credential,
    )
    agent = chat_client.as_agent(
        name="TravelAgent",
        instructions=(
            "You are a helpful travel agent. Help users find vacation destinations that match their preferences. "
            "Use the get_destinations tool to see available options and provide personalized recommendations."
        ),
        tools=[get_destinations],
    )
    tool_mode_enabled = True
    print("✓ Agent created successfully")
except Exception as e:
    print(f"✗ Failed to create client/agent: {type(e).__name__}: {str(e)[:200]}")
    raise

# Run the agent
print("\n" + "="*60)
print("Running Travel Agent")
print("="*60)

user_prompt = "I'm looking for a warm beach destination. What do you recommend?"
print(f"\nUser: {user_prompt}")

try:
    result = await agent.run(user_prompt)
    print(f"Agent: {result.text}")
except Exception as e:
    err_text = str(e)
    if "ApiSamplingErrorUnprocessableInput" in err_text or "Invalid parameter" in err_text:
        print("\n⚠️ This deployment does not support tool-calling parameters in this flow.")
        print("Falling back to no-tool mode so the lesson can keep running.")

        # Fallback: re-create agent without tools for model compatibility.
        agent = chat_client.as_agent(
            name="TravelAgentNoTools",
            instructions="You are a helpful travel agent. Recommend warm beach destinations based on user preferences.",
        )
        tool_mode_enabled = False
        fallback_prompt = (
            f"Use this available destination list as context: {', '.join(get_destinations())}. "
            f"User request: {user_prompt}"
        )
        result = await agent.run(fallback_prompt)
        print(f"Agent (fallback no-tool mode): {result.text}")
        print("\nTip: set AZURE_AI_MODEL_DEPLOYMENT_NAME to a tool-capable model (for example gpt-4.1-mini or gpt-4o-mini).")
    else:
        print(f"✗ Error running agent: {type(e).__name__}: {e}")
        raise

Creating FoundryChatClient and Agent...
Using model deployment: gpt-oss-120b
✓ Agent created successfully

Running Travel Agent

User: I'm looking for a warm beach destination. What do you recommend?

⚠️ This deployment does not support tool-calling parameters in this flow.
Falling back to no-tool mode so the lesson can keep running.
Agent (fallback no-tool mode): Here are a few warm‑beach spots from the list you provided, along with what makes each one special. Let me know which vibe sounds most appealing (or if you have a particular season or activity in mind) and I can fine‑tune the recommendation!

| Destination | Typical Beach Weather (°C / °F) | Highlights | Best Time for Warm Beaches |
|-------------|----------------------------------|------------|-----------------------------|
| **Rio de Janeiro, Brazil** | 25‑30 °C / 77‑86 °F (summer Dec‑Mar) | Iconic Copacabana & Ipanema, vibrant nightlife, Rio’s Carnival, Sugarloaf Mountain and Christ the Redeemer nearby. | December – March 

## Microsoft Agent Framework Pattern

The current Microsoft Agent Framework flow that is compatible with this environment uses `FoundryChatClient`:

```python
from azure.identity.aio import AzureCliCredential
from agent_framework_foundry import FoundryChatClient
from agent_framework import tool

@tool(approval_mode="never_require")
def my_tool() -> str:
    """Tool description for the model"""
    return "result"

chat_client = FoundryChatClient(
    project_endpoint=os.getenv("AZURE_AI_PROJECT_ENDPOINT"),
    model=os.getenv("AZURE_AI_MODEL_DEPLOYMENT_NAME"),
    credential=AzureCliCredential(),
)

agent = chat_client.as_agent(
    name="MyAgent",
    instructions="Your agent's system prompt",
    tools=[my_tool],
)

result = await agent.run("user message")
print(result.text)
```

Behind the scenes, the framework still handles orchestration and tool invocation for you.

In [3]:
# Ask a second question
print("\n" + "="*60)
user_prompt = "I prefer culture and food over beaches. Which destination should I pick?"
print(f"User: {user_prompt}")

if 'tool_mode_enabled' in globals() and not tool_mode_enabled:
    follow_up_prompt = (
        f"Use this available destination list as context: {', '.join(get_destinations())}. "
        f"User request: {user_prompt}"
    )
    result = await agent.run(follow_up_prompt)
    print(f"Agent (fallback no-tool mode): {result.text}")
else:
    result = await agent.run(user_prompt, options={"model": model_name})
    print(f"Agent: {result.text}")


User: I prefer culture and food over beaches. Which destination should I pick?
Agent (fallback no-tool mode): **Top Recommendation: Tokyo, Japan**

**Why it fits your love of culture and food**

| Feature | What You’ll Experience |
|---------|------------------------|
| **Culinary heaven** | From Michelin‑star sushi and ramen shops to street‑food stalls in Asakusa, Tokyo offers an unmatched variety of flavors. You can try everything from traditional kaiseki (multi‑course meals) to regional specialties like Osaka‑style takoyaki and Hokkaido seafood—all within the city. |
| **Rich cultural heritage** | Explore historic neighborhoods (Asakusa’s Senso‑ji Temple, the Imperial Palace grounds), world‑class museums (Mori Art Museum, Edo‑Tokyo Museum), and traditional arts such as kabuki theater and tea ceremonies. |
| **Modern meets traditional** | Wander neon‑lit Shibuya, futurist Odaiba, and the fashion district Harajuku, then step back in time at Meiji Shrine or the historic Yanaka distric

## Summary

In this lesson you learned:

- Use `FoundryChatClient` to connect to Azure AI Foundry with your model deployment.
- Define tools with the `@tool` decorator to expose functions to your agent.
- Wrap the client as an agent with `chat_client.as_agent(...)`.
- Run the agent with `await agent.run()` and read `result.text`.
- Ask follow-up questions using the same agent instance.
- Handle model compatibility constraints by falling back to a no-tool path when tool-calling parameters are not supported.

The key insight: **You focus on tools and instructions, while the framework handles orchestration and tool-calling flow.**

Before running, ensure:
- `az login` has been run to authenticate with Azure
- `AZURE_AI_PROJECT_ENDPOINT` (or `FOUNDRY_PROJECT_ENDPOINT`) is set
- `AZURE_AI_MODEL_DEPLOYMENT_NAME` is set to your deployed model name
- Your Azure CLI is set to the correct subscription with your AI Foundry project

If you want full tool-calling behavior, use a compatible deployment such as `gpt-4.1-mini` or `gpt-4o-mini`.